In [ ]:
################################################
# New style 
################################################
import sys
import os

workdir_ = '/glade/work/juliob'
if ( workdir_ not in sys.path ):
    sys.path.append(workdir_)
    print( f" a path to {workdir_} added in {__name__} ")

from myPythonTools.Utils import utils as uti
from myPythonTools.Utils import numerical_utils as nuti
from myPythonTools.Utils import AveragingUtils as Av
from myPythonTools.Utils import validation_data as Val
from myPythonTools.Utils import PlotUtil as Pu
from myPythonTools.Plotting import LatLonMaps as LL 
from myPythonTools.Plotting import utils as puti
from myPythonTools.CASutils import filter_utils as fu 

from PyRegridding.Utils import GridUtils as GrU
from PyRegridding.Utils import MakePressures as MkP
from PyRegridding.Drivers import RegridField as RgF


# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.tri as tri

# Cartopy for pretty maps
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Some other useful packages 
import importlib
import copy
import time
import cftime

importlib.reload( uti )
importlib.reload( puti )
importlib.reload( Pu )
importlib.reload(Av)
importlib.reload(Val)
#importlib.reload(vAB)
importlib.reload(MkP)
importlib.reload(RgF)


In [ ]:

#WACCM forcing used for radmods ....
f='/glade/campaign/cesm/cesmdata/cseg/inputdata/atm/waccm/waccm_forcing/SCWACCM_forcing_zm_L70_1849-2015_CMIP6ensAvg_c181011.nc'
X=xr.open_dataset( f )

In [ ]:
X.QRS_TOT


In [ ]:
print( X.lat.values )

In [ ]:
qrs_tot=X.QRS_TOT.values
lat=X.lat.values
zlev = -7.0*np.log(X.lev.values/1_000.)


In [ ]:
qrs_tot=qrs_tot.reshape( 228//12,12,70,192)

In [ ]:
qrs_tot.shape

In [ ]:
qrs_tot_cyc = np.mean( qrs_tot, axis=0 )
qrs_tot_av = np.mean( qrs_tot_cyc, axis=0 )
qrs_tot_ann = np.mean( qrs_tot, axis=1)


In [ ]:
plt.plot( qrs_tot_ann[:,1,96] )
plt.plot( qrs_tot_ann[:,2,96] )
plt.plot( qrs_tot_ann[:,5,96] )

In [ ]:



fig,axs=plt.subplots( 1,3, figsize=(16,5))
ax=axs[0]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_av[:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
cbar=plt.colorbar(co)
ax=axs[1]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_cyc[2,:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
ax=axs[2]
co=ax.contourf(lat,zlev, 86_400.*qrs_tot_cyc[6,:,:], levels=1*np.linspace(0,10,num=11) )
ax.set_ylim(0,95)
#plt.colorbar()
#plt.plot( X.QRS_TOT[0,:,7] * 86_400. ,-7*np.log(X.lev/1_000.) )




In [ ]:
plt.plot( 86_400.*qrs_tot_cyc[6,55,:] )

In [ ]:
plt.plot( 86_400.*qrs_tot_av[:,95] ,zlev )
plt.ylim(0,100)
plt.xlim(0,20)
plt.ylabel('Z')

In [ ]:
print(zlev[10])

In [ ]:
def f_LTE(p_hPa, p0=1.0, k=1.6):
    """
    Fraction of short-wave heating that behaves as LTE, as a function of pressure (hPa).
    p0 ≈ transition pressure (~1 hPa ≈ 50 km).
    k  controls sharpness (1.3–2.0 reasonable).
    Returns values in [0,1].
    """
    p = np.maximum(p_hPa, 1e-6)
    return 1.0 / (1.0 + (p0 / p)**k)


flte = f_LTE( X.lev[10:] ,p0=0.5 )

In [ ]:
#import numpy as np
from scipy.optimize import curve_fit

def model0(z, c0, c1, a1, z1, s1, a2, z2, s2):
    return c0 + c1*z + a1*np.tanh((z - z1)/s1) + a2*np.tanh((z - z2)/s2)

def model1(z, c0, c1, a1, z1, s1):
    return c0 + c1*z + a1*np.sin((z - z1)/s1)

p0 = [0.3, 0.07, 2.0, 15.0, 6.0, 1.8, 75.0, 8.0]  # start guesses
p1 = [0.3, 0.07, 1.0, 45.0, 6.0 ]  # start guesses

x0=86_400.*qrs_tot_av[10:,95]
z0=zlev[10:]
# z, x = your data arrays used to make the plot
popt, pcov = curve_fit(model0, z0, x0, p0=p0, maxfev=20000)


x=(1.0-flte)*x0
x=x[0:21]
z=z0[0:21]
z_meso = z0[0:11]
popt1, pcov1 = curve_fit(model1, z, x, p0=p1, maxfev=20000)

# popt holds best-fit parameters


In [ ]:
c0_0, c1_0, a1_0, z1_0, s1_0, a2_0, z2_0, s2_0 = popt
c0_1, c1_1, a1_1, z1_1, s1_1 = popt1
print( c0_1, c1_1, a1_1, z1_1, s1_1 )

In [ ]:
plt.plot( x0,z0 , label='mean WACCM QRS_TOT Forcing',color='blue' )
plt.plot( model1(z_meso, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z_meso , label='functional fit 2' , color='red', lw=3 )
plt.plot( model1(z0, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z0, '--',label='functional fit 2, complete range' ,color='red')
plt.xlabel('Q (K d$^{-1}$)' )
plt.ylabel('Z (km)' )
plt.legend()
print( z_meso )

In [ ]:
plt.plot( x0,z0 , label='mean WACCM QRS Forcing' )
plt.plot( model0(z0, c0_0, c1_0, a1_0, z1_0, s1_0, a2_0, z2_0, s2_0) , z0, label='functional fit 1' )
plt.plot( model1(z, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z, '-o', label='functional fit 2' )
plt.plot( model1(z0, c0_1, c1_1, a1_1, z1_1, s1_1 ) , z0, label='functional fit 2, complete range' )
plt.xlabel('Q (K d$^{-1}$)' )
plt.ylabel('Z (km)' )
plt.legend()


In [ ]:
print( popt )

In [ ]:
x0=86_400.*qrs_tot_av[10:,95]
z=zlev[10:]
#nlte_frac=np.exp( z / 70. )


In [ ]:

plt.plot( (1.0-flte)*x0 , z)
plt.plot( x0 , z)


In [ ]:
#plt.contourf(X.QRS_TOT[0,:,:] )
plt.plot( X.QRS_TOT[0,:,7] * 86_400. ) #,-7*np.log(X.lev/1_000.) )


In [ ]:
plt.plot( X.QRS_TOT[:,15,7] * 86_400. )

In [ ]:
plt.tricontour(X.lon,X.lat,X.LANDFRAC,levels=[0.5,10.])
plt.xlim( 250,320 )
plt.ylim( 10,45)

In [ ]:
f='/glade/campaign/cesm/cesmdata/inputdata/atm/cam/chem/ubc/b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.ensAvg123.cam.h0zm.H2O.185001-201412_c230509.nc'

dir='/glade/derecho/scratch/juliob/radtest_Q02_mpas120_fmt_c6477/run/'
f=dir+'radtest_Q02_mpas120_fmt_c6477.cam.h1i.1985-01-01-01800.nc'

dir='/glade/derecho/scratch/juliob/radtest_AICN01hf_mpas120_fmt_c6477/run/'
f=dir+'radtest_AICN01hf_mpas120_fmt_c6477.cam.h1i.1985-01-05-46800.nc'
X=xr.open_dataset( f )

In [ ]:
X


In [ ]:
#te=X.T.values
lon=X.lon.values
lat=X.lat.values
u=X.U.values
utg=X.Target_U.values




In [ ]:
print(np.shape(u))
ulev=np.linspace( -200,200,num=51)

In [ ]:
plt.tricontourf( lon, lat, u[0,0,:], cmap='bwr_r' ,levels=ulev)
plt.colorbar()

In [ ]:
plt.plot(u[0,:,:].flatten())
#plt.plot(te3[0,0,:])


In [ ]:
plt.plot(te[0,0,:])
plt.plot(te3[0,0,:])
